# Assignment 2: Eclipsing Binary Star Populations

Jason, Luca, Ian

In this tutorial, we will demonstrate how to generate a realistic sample of binary star systems, and from this, we will aim to determine a population of eclipsing binary stars and investigate how the individual parameters affect the likelihood of the system eclipsing. The above image shows the elliptic motion of a binary star system and the angular parameters that must be considered. Specifically, the parameters that we will focus on in this assignment are primary mass \($M_{\odot}$\), mass ratio \(q\), period \(P\), eccentricity \(e\), angle of inclination \(I\), and two angular parameters given by Ω and ω.

We will split this tutorial into 5 different chapters:

Chapter 1: CDF Creation

Chapter 2: Generating Realistic Samples for Each Individual Parameter

Chapter 3: Generating Samples of Binary Star Systems

Chapter 4: Detecting Eclipses

Chapter 5: Comparing Eclipsing Population versus Binary Star Population and Discuss Results

Chapter 6: Comparing Primary Eclipses and Secondary Eclipses

By the end of this tutorial, you should have zero problems generating realistic, awesome\-looking binary star systems! 


<img src="eclipse.jpg" width="400" alt="Solar eclipse"/>

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sci

# Chapter 1: CDF Creation

In creating a population of binary systems we want to be sure our sample is representative of real, physical systems. This means we want our sample to replicate the distributions of crucial parameters such as stellar mass, period of orbit, eccentricity, and 3D angles. Some of these distributions, such as stellar mass, are not trivial so generating a list of N random values that maps to the realistic spread of stellar masses will be tricky. For this project we will utilize CDFs, Cumulative Distribution Functions, to quickly create large numbers of random systems. A CDF, in principle, is a function $\text{CDF}(x)$ that tells us about the integral of some other function $f(x')$ between it's minimum value $x_0$ and $x$.

$\text{CDF}(x) = \int_{x_0}^x f(x') dx'$

Where x' here simply serves as a dummy variable.

Here's how a CDF can benefit population sythesis. Say we make the function ($f(x')$) in the integrand the PDF, the Probability Density Function, where the value of $\text{PDF}(x)$ is the probability of some value $x$ is true within an infinitesimally small bin, dx. The CDF in this case tells us the cumulative probability that some variable will be found between the minimum possible value $x_0$ and $x$. It's important to note the probability can only range from 0 to 1 (in other words, our PDF is normalized and the integral of the PDF is equal to one) and so for any PDF the CDF will range from 0 to 1.

Now here's a cool trick. If we take the CDF of our PDF (PDF-CDF one may say) and generate a random value between 0 and 1, we can set that to be a value on the y axis of the CDF. If we find what value on the x-axis corresponds to this random CDF value we can note it and make that x value a part of our sample. If we do this N times the list of N value for the parameter x will mirror the PDF. One way to see why this works is to look at the above figure. From the PDF we expect a larger amount of our values to be generated around the central value of x, note that this same region corresponds to an area of the CDF with a high slope. In essence, this small, central region of the CDF is covering more of the availble y space from 0 to 1 so if a value is generated then it's more likely to be from the central area than either of the wings. So this method of random sampling is robust in making samples as every generated value can be used in our sample (unlike acceptance/rejection sampling). We just need to make our CDFs beforehand and call on them when we want to start generating our sample.

The code below is designed to return a full CDF as an array for any pdf passed as an array. It leverages the idea that the PDF is passed as an array and each value of the array is separated by some distance on the x-axis to create a Riemann sum. To make it work you must pass a PDF an array of x-values (called x_space) that is the same length as the PDF array. By "x-values" I'm simply referring to any parameter you're plotting on the x axis of a distribution. For our cases these are the parameters of binary systems such as mass, radius, period, and eccentricity. You want it such that the values of x_space correspond to the values of the PDF array such that the value PDF[0] corresponds to the x-value indexed at 0, x_space[0] and so on.

Imagine the above figure is our PDF. The solid line represents the analytical form of the PDF that is a function while the bars represent the PDF array we are passing where the height is the value of the array at some index. The width of the bar ($\Delta x$) corresponds to the separation of the values of x_space. For example, if we have a hypothecal PDF that takes values on the x axis ranging from 0 to 1 and we split that into ten bins equally spaced then we will have 10 values of the pdf that are each spaced by 0.1 on the x axis. Because a PDF is normalized if we sum up the areas of these bars they should add up to 1.

We start from the minimum x-value of our distribtion and start integrating towards the maximum x-value. To do this, initialize some variable, $\texttt{sum}$, to be zero. We will then add the areas of the rectangles iteratively to $\texttt{sum}$. Recall the area of the rectangle is equal to the height of the rectangle which is the value of the pdf times the width which is the difference of array values of x_space. This process is just a Riemann sum, and for sufficiently small bins (equivalent to saying we have a large enough number of values of the PDF and x_space arrays) this should approximate the integral very well.

Now we recall the definition of a CDF at some value $x_i$ is value of the integral starting at the minimum x-value, $x_0$ to the value $x_i$:

$\text{CDF}(x_i) = \int_{x_0}^{x_i} \text{PDF}$

So this means that as we iterate along the areas of the rectangles and add the values to $\texttt{sum}$ we are tabulating the values of the CDF of all the values of x. This means when we add the area of the first rectangle (the one between $x_0$ and $x_1$ in the figure above) to $\texttt{sum}$ we've calculated the CDF at $x_1$ (the value of the CDF for $x_0$ must be zero). When we add the area of the second rectangle to that running tally of $\texttt{sum}$ then we get the sum of the areas of both the first and second rectangle, meaning we've integrated from $x_0$ to $x_2$, equivalent to the CDF at $x_2$. So long as we can store these values as we integrate from $x_0$ to the max value $x_{\text{max}}$ we should have our list of CDF values when we're done that's the same length as the PDF and x_space arrays.


In [5]:
#CDF Generator, should work for any normalized distribution function

#The function will return the full CDF from the minimum allowed
#value in the distribution to the maximum

#x defines the upper bounds of the integration from zero

#Until the upper bound x is reached

#Input distribution, dist, must be inserted as 1D array

#x_space is the 1D array of elligible values in the distribution

#x_space should be same shape as dist!!

def CDF_generator(pdf, x_space):

    #Initialize the value of the CDF
    #This will keep climbing towards 1 as we iterate
    #And sum up each little bit of the pdf
    CDF = 0

    #Initialize the list that will store the values of the CDF at each x value
    #As we move along the distribution we'll keep track of the CDF values and store them
    #Here

    #It will end up being the same length as the x_space argument so we'll know each value
    #of the cdf corresponds to a certain value in the distribution

    #Okay so some rapidly rising CDFs like the IMF have this fun feature where they rise so
    #fast that their first indexed value is pretty large (like 0.038), this means that when inverting
    #the CDF there's a 3.8% chance the random number generator picks a value for which the invertor can't
    #find an appropriate value on the x-axis (there's no CDF value for the algorithm to find if the random
    #number generator picks 0.02) so we get an infinite loop. To fix this we'll assert that the first value of
    #the CDF must be exactly zero so we do have values of the CDF rangning from 0 to 1
    
    #Keep in mind that later on we may get the reverse problem! We to make sure that the last indexed
    #value of the CDF is 1 or really close to one
    
    CDF_list = [0]

    i = 0

    #We'll keep iterating until we integrate over the full PDF
    #the length must be the length of the pdf minus one because we already filled
    #the first index with zero
    while i < (len(pdf) - 1):
        
        #Add the area of one bin of the pdf to the CDF total
        
        #I chose to make the width x_space[1] - x_space[0]
        #as to be invariant to what binning I'm using
        CDF = CDF + (pdf[i]* (x_space[1] - x_space[0]))
        
        #Append that CDF value to the array

        CDF_list.append(CDF)

        i = i + 1
        
    #make sure the last index is equal to 1 so we don't generate a random number
    #outside the range of the y axis of the CDF
    
    CDF_list[len(x_space) - 1] = 1

    return CDF_list

## CDF Inversion

Now I'm going to show how to make a generalized CDF inverter to turn any y value of any CDF into a value on the x-axis. It's a bit trickier than first expected. The first step is to generate a CDF from a PDF using the previously defined function. The first catch in this process is to recall that we want to find the value along the x axis that when plugged into the CDF returns a desired y value (we just call that input y in our code). It turns out this is tough to do if we leave our CDF as an array because it's tricky to correlate a specific value on the x-axis with an index in the array. The best thing to do is to turn the CDF into something callable where we can pass in any possible x value and get out the y value of the CDF there.

Interpolating the CDF is the best way to accomplish this. If you don't know what interpolation is you should add it to your tool belt for sure. scipy.interpolate.interp1d will take a list of x values and list of y values of equal length (which is the CDF in this case) and connect the values so every possible x value has a y value. It does this by just drawing a straight line from one point to the other so if your data has curves you need to make sure you are providing a sufficient number of points for the interpolation to be an accurate approximation, although this is rarely an issue in practice.

We don't need anything fancy, linear interpolation should work just fine.

With this interpolated callable CDF we can run a binary search. If you tried the guessing challenge from the first week of class you may already be familiar with the process. Essentially we start in the middle of the x-sample range and check to see in the CDF there is higher or lower than the y value we want. If it's higher then we know we need to search the first half of the range so we redo the search there looking halfway between the lowest possible value and the value we just guessed. If the guess was too low we look in the second half of the range. Repeat the process until the value of the CDF at some x coordinate is within some tolerance from the desired y value. When that's done we want to return the x value.

For the following function we will have it such that we need to pass a desired value on the y axis, an INTERPOLATED CDF, and the list of x vlaues. The first input gives the code a goal to shoot for, the second is the CDF it will trace to try to find the y value, and the x_space is just to initialize the guess of binary search.


In [6]:
#Find x value for some y value
#I'm going to use binomial search because the fucnction monotomically rises

def CDF_invert(y, cdf_interp, x_space):
    
    #max guessing bound, will change during the loop
    #start it at the maximum possible value of the x_values
    maxi = max(x_space)

    #min guessing bound, will change during the loop
    #start at the minimum possible value of the x_values
    mini = min(x_space)

    #initialize the guess for the right value in the middle
    #of the range
    x = 0.5 * (mini + maxi)

    #keep iterating until our CDF value reaches desired value plus or minus a small tolerance value
    while (cdf_interp(x) > y + 0.0001) or (cdf_interp(x) < y - 0.0001):

        #if the CDF here is too big
        if cdf_interp(x) > y:

        #Change the guessing range to be between mini and x
            maxi = x
        
        #Update the value of the guess to be halfway between mini and what
        #we just guessed
            x = 0.5 * (mini + x)

        
        #If the CDF is too small
        if cdf_interp(x) < y:

        #Change the guessing range to be between maxi and x
            mini = x
        
        #Update the value of the guess to be halfway between maxi and what
        #we just guessed
            x = 0.5 * (maxi + x)

    #Tell us what x value caused the loop to break    
    return x

## Generating Random Samples

Now we generate a bunch of random values for our sample using the CDF we've defined earlier. The process will involve generating a number between zero and one and using the CDF inverter to see what x value this corresponds to. It's pretty straightforward and simple, just make a random value of the CDF, plug it into the CDF invertor, log the result, repeat N times.



In [7]:
def sample_generator(N, cdf_interp, x_space):

    #initialize empty list of values for the sample

    sample_list = []

    #start the loop, which will repeat N times

    i = 0

    while i < N:

        #generate random y value of the CDF

        y = np.random.rand()

        #see what x value that represents from the inversion

        sample_value = CDF_invert(y, cdf_interp, x_space)

        #add that value to the list for our random sample

        sample_list.append(sample_value)

        #Move to next iteration
        i = i + 1

    return sample_list

# Chapter 2: Generating Samples for Each Parameter



## Generating Period \(P\)

### Period CDF

The period distribution of binaries is given with a log\-normal distribution with mean μ\(logP\) = 5.03 and standard deviation 𝜎 \(logP\) = 2.28, both in days. This means we'd get a normal distribution but only if the x axis is in log scale so we need to be extra careful about doing this. To make this easier to code I'm going to mostly be working in log\(P\) as my key variable and convert it to period when I absolutely have to. For anyone reading I highly suggest getting comfortable working both in log and linear scaled units because both are commonly used in astronomy.



In [8]:
# Generate a normal distribution using scipy's normal pf

#Set the mean, μ, in log space. Time measured in days
log_mu = 5.03

#Set the std deviation, σ, in log space. Time measured in days.
log_sigma = 2.28

#Set range of possible periods within 2.5σ of the mean
log_P = np.linspace(log_mu - 2.5 * log_sigma, log_mu + 2.5 * log_sigma, 1000) #log(period) in days 

#generate pdf object using a functional form of normal dist
#again, just leaving everything in log space makes everything much easier
def normal_dist(x_space, mu, sigma):
    y = 1/(sigma * (2 * np.pi)**0.5) * np.exp(-1*(x_space - mu)**2/(2*sigma**2))
    return y

#generate pdf values
log_P_pdf = normal_dist(log_P, log_mu, log_sigma)

#plot

plt.figure(figsize = (10, 8))
plt.plot(log_P, log_P_pdf)
plt.xlabel(r'log(P [days])', fontsize = 20)
plt.ylabel(r'PDF', fontsize = 20)
plt.tick_params(direction = 'in', labelsize = 16)
plt.xlim(min(log_P), max(log_P))
plt.show()

Good news is a PDF is already normalized. Recall the PDF stands for Probability Density Function so when we integrate it over all possible values we get 1, the probability any allowed value is chosen. Since the range within 2.5σ of the mean comprises > 95% of the data we can be sure this will approximately integrate to one. Now we just need to build our CDF and invert it, which we did in class already!



In [9]:
#values of log P allowed
log_P = np.linspace(log_mu - 2.5 * log_sigma, log_mu + 2.5 * log_sigma, 1000)

#Generate Period CDF from the PDF

log_P_cdf = CDF_generator(log_P_pdf, log_P)

#Interpolate the CDF to smooth it out and make an x value for
#any possible input x value}

#It's also just great to make the CDF a callable rather than
#an array so for each guess for the x value we can just plug it
#in rather than trying to figure out what index it corresponds to
#in an array

log_P_cdf_interp = sci.interpolate.interp1d(log_P, log_P_cdf)


#PLOT
#Always make sure your plot is big with large text, I'm trying to not lose my sight
#from constant squinting at my blue light death machine.
plt.figure(figsize = (10,8))
plt.plot(log_P, log_P_cdf)
plt.xlabel('log(P [Days])', fontsize = 20)
plt.ylabel('CDF', fontsize = 20)
plt.tick_params(direction = 'in', labelsize = 16)
plt.xlim(min(log_P), max(log_P))
plt.show()

In [10]:
#Plot sample generation

sample_log_P = sample_generator(1000, log_P_cdf_interp, log_P)

plt.figure(figsize = (10, 8))
plt.hist(sample_log_P, density = True, label = 'Generated Sample')
plt.plot(log_P, log_P_pdf, color = 'black', linestyle = '--', label  = 'Analytical PDF')
plt.xlabel('log(P [Days])', fontsize = 20)
plt.ylabel('PDF', fontsize = 20)
plt.tick_params(direction = 'in', labelsize = 16)
plt.legend(fontsize = 16)
plt.show()

And hence, our period \(P\) sample generator is completed! However, we must remember to convert this back into Period \(days\) from log P, and to always be aware of **units**!


## Generating Primary Mass $M_{\odot}$

### Salpeter IMF

The Salpeter IMF determines the relative occurrence of stellar masses of our sample. The formula is given as:

$\xi \propto M^{\alpha}$ where for the Salpeter IMF $\alpha = -2.35$

$\xi$ is kind of weird and has units of $\frac{1}{dM}$ where dM is the bin of stellar mass in solar masses. This is because the exact value of $\xi$ depends on the binning on the x-axis. But essentially the way to read an IMF graph is that the y axis tells us the relative occurrence of stars and the x axis is the mass of the stars. So the higher up you are the more common the star is. It's also common to plot these IMF graphs in log space (log(M) vs log($\xi$)) because power laws become nice diagonal lines in a log-log graph.



In [11]:

#define Salpeter +55 power law IMF

def Salpeter(M):
    alpha = -2.35
    return M**alpha

#generate sample of masses
M = np.linspace(0.1, 3, 500)


#plot in log-log space, typically used for IMFs to see that sweet sweet straight line
plt.figure(figsize = (10, 8))
plt.plot(np.log10(M), np.log10(Salpeter(M))) 
plt.xlabel(r'log(M [$M_{\odot}]$)', fontsize = 20)
plt.ylabel(r'$log(\xi) \frac{dN}{d(logM)}$', fontsize = 20)
plt.tick_params(direction = 'in', labelsize = 12)
plt.xlim(-1, 0.47)
plt.show()


#plot in linear space
plt.figure(figsize = (10, 8))
plt.plot(M, Salpeter(M)) 
plt.xlabel(r'M [$M_{\odot}]$', fontsize = 20)
plt.ylabel(r'$\xi (\frac{dN}{dM})$', fontsize = 20)
plt.tick_params(direction = 'in', labelsize = 12)
plt.xlim(0.1, 3)
plt.show()

### Normalize it

In order to normalize this graph we simply need to take the array of values we plugged into the y axis, Salpeter(M), and divide it by the area under the curve. We can take a basic Reimann sum to do this, the sum of the area of rectangles times the widths of each one. This is equal to the sum of the values of the indeces times the width of bins.

$\int_{x_0}^{x_{max}} PDF = PDF[0]* \text{bin\_width} + PDF[1] * \text{bin\_width}...$ 

$\int_{x_0}^{x_{max}} PDF = \sum_{i = 0}^{i = N} PDF[i] * \text{bin\_width} =  \text{bin\_width} * \sum_{i = 0}^{i = N} PDF[i]$

Note: $\sum_{i = 0}^{i = N} PDF[i] = \text{sum(PDF)}$ in Python


The resulting array will be normalized and integrate to unity. This is a good thing to keep in mind if you ever need to normalize a function or distribution and is exactly what python does when you turn on the density keyword argument in plt.hist() or np.histogram().

One important thing to note is that the exact value of the normalized array depends on the number of elements in the array. The more elements in the array the smaller value each one will have once normalized. So two people may have slightly different PDF values of their distributions not because one of them did something wrong but because their x_space samples were different sizes. I doubt it will cause much concern because the CDF should still come out the same but it's good to know in case you need to debug.

In [12]:
#Generate x_space for masses from 0.1 to 3 solar masses

M = np.linspace(0.1, 3, 1000)


#generate intial array of IMF values
salpeter_pdf = Salpeter(M)

#normalize it by dividing the array by the sum of its elements

salpeter_pdf = salpeter_pdf / ((M[1] - M[0]) * sum(salpeter_pdf))

#plot

plt.figure(figsize = (10, 8))
plt.plot(M, salpeter_pdf) 
plt.xlabel(r'M [$M_{\odot}]$', fontsize = 20)
plt.ylabel(r'$\xi (\frac{dN}{dM})$', fontsize = 20)
plt.tick_params(direction = 'in', labelsize = 12)
plt.xlim(0.1, 3)
plt.show()

This may looks strange because some of the y values are larger than 1 but keep in mind the bin widths are <<1 so the area of each bin is still very small even if it is tall.

### Check the CDF

In [13]:
#Generate x_space for masses from 0.1 to 3 solar masses

M = np.linspace(0.1, 3, 1000) #in solar masses

M_bin_width = (M[1] - M[0]) #width of bins of solar mass, equally spaced


#generate intial array of IMF values
salpeter_pdf = Salpeter(M) 


#normalize it by dividing the array by the sum of its elements
salpeter_pdf = salpeter_pdf / (M_bin_width * sum(salpeter_pdf))

#generate the CDF
Salpeter_cdf = CDF_generator(salpeter_pdf, M)

#PLOT
#Always make sure your plot is big with large text, I'm trying to not lose my sight
#from constant squinting at my blue light death machine.
plt.figure(figsize = (10,8))
plt.plot(M, Salpeter_cdf)
plt.xlabel('M', fontsize = 20)
plt.ylabel('CDF', fontsize = 20)
plt.tick_params(direction = 'in', labelsize = 16)
plt.xlim(0.1, 3)
plt.show()

### Generate our random sample of stellar masses



In [14]:
#Generate x_space for masses from 0.1 to 3 solar masses

M = np.linspace(0.1, 3, 1000) #in solar masses

M_bin_width = (M[1] - M[0]) #width of bins of solar mass, equally spaced


#generate intial array of IMF values
salpeter_pdf = Salpeter(M)


#normalize it by dividing the array by the sum of its elements
salpeter_pdf = salpeter_pdf / (sum(salpeter_pdf) * M_bin_width) #This one is normalized


#Generate the CDF
Salpeter_cdf = CDF_generator(salpeter_pdf, M)


#Interpolate it
Salpeter_cdf_interp = sci.interpolate.interp1d(M, Salpeter_cdf)


#Make the sample using the sample generator
sample_M = sample_generator(1000, Salpeter_cdf_interp, M)

#plot the distribution

plt.figure(figsize = (10, 8))
plt.hist(sample_M, density = True, label = 'Generated Mass Sample', bins = 20)
plt.plot(M, salpeter_pdf, color = 'black', linestyle = '--', label  = 'Salpeter PDF')
plt.xlabel(r'$M (M_{\odot})$', fontsize = 20)
plt.ylabel('PDF', fontsize = 20)
plt.tick_params(direction = 'in', labelsize = 16)
plt.legend(fontsize = 16)
plt.xlim(0.1, 3)
plt.show()

## Generating Ω

For the purposes of this project we will be calculating the eclipses along the $\hat{X}$ direction from the first figure of this notebook. It's defined as the reference direction and lies in the reference plane in the first figure of this notebook. Becasse we're doing something slightly different than what was done previously in the 3d_orbits notebook we need to see if and how Ω will affect the results. In the original notebook, Ω has no effect because all motion was calculated with the point of view being along the $\hat{Z}$ axis. This means all apparent motion was in the XY plane, the same plane in which the Ω angle rotates the orbit. This means Ω won't be able to move the secondary into or out of the line of sight to the primary so it won't affect the eclipsing. However, we will do something slightly different, where Ω will be able to move the secondary star into or out of the line of sight to the primary. We expect eclipsing to only really occur where Ω is approximately 0, π, or 2π since that would put the ascending node right on the $\hat{X}$ axis.

Since Ω is uniform between 0 to 360 degrees, or 0 to 2π radians, we can use the np.random.uniform function from numpy to make it a lot simpler!


In [15]:
# Allow the Greek letter Ω to be shown in labels, titles and axes
cap_omega = "\u03A9"

# Since Ω is also uniform between 0 and 360, we can use np.random.uniform() again
Omega = np.random.uniform(0, 2 * np.pi,1000)

# Create an array of the same shape as Ω where every element is filled with the value
uniform_pdf = np.full_like(Omega, 1/(np.pi * 2))  

# Plot the uniform PDF and generated Ω samples
plt.figure(figsize=(10, 8))
plt.hist(Omega, density = True, label = 'Generated '+ cap_omega +' Sample', bins = 20)
plt.plot(Omega, uniform_pdf, color='black', linestyle = '--', label='Uniform PDF for ' + cap_omega)
plt.xlabel(cap_omega + '[Radians]', fontsize=20)
plt.ylabel('PDF', fontsize=20)
plt.tick_params(direction='in', labelsize=16)
plt.xlim(0, np.pi * 2)
plt.legend(fontsize=16)
plt.title('Uniform Distribution PDF for ' + cap_omega, fontsize=20)
plt.show()

## Generating 𝜔, longitude of pericentre

A similar case goes for 𝜔, the other angular parameter that ranges uniformly from 0 to 360 degrees, or 0 to 2π radians, meaning we can use np.random.uniform\(\)


In [16]:
# Allow the Greek letter 𝜔 to be shown in labels, titles and axes
low_omega = "\u03C9"

# Since 𝜔 is uniform between 0 and 360, we can use np.random.uniform()
omega = np.random.uniform(0, 2 * np.pi,1000)

# Create an array of the same shape as 𝜔 where every element is filled with the value
uniform_pdf = np.full_like(omega, 1/(2 * np.pi))  

# Plot the uniform PDF and generated 𝜔 samples
plt.figure(figsize=(10, 8))
plt.hist(omega, density = True, label = 'Generated '+ low_omega +' Sample', bins = 20)
plt.plot(omega, uniform_pdf, color='black', linestyle = '--', label='Uniform PDF for ' + low_omega)
plt.xlabel(low_omega + '[Radians]', fontsize=20)
plt.ylabel('PDF', fontsize=20)
plt.tick_params(direction='in', labelsize=16)
plt.xlim(0, np.pi * 2)
plt.legend(fontsize=16)
plt.title('Uniform Distribution PDF for Longitude of Pericentre ' + low_omega, fontsize=20)
plt.show()

## Generating Inclination \(I\)

However, the angle of inclination \(I\) is a little different, and a little more tricky. While the past two angles have been uniform, the inclination works a little differently. Imagine a unit hemisphere centered on the focus of an orbit and oriented towards the observer:

We can represent the inclination of the orbit as being the angle between the line of sigh of the observe and some vector that connects the center of hemisphere to a point on its surface. These vectors will be distributed isotropically so that no one direction is favored. The number of vectors that correspond to a given inclination, $i$, is proportional to the area shell (in gray) of the surface of the hemisphere that houses vectors with an angle of inclination within an infinitesimally small width, $di$. These areas are not equal for different inclinations, so some inclinations closer to the base of the hemisphere will be more likely to occur than other nearer to the pole. If you squint at it you might be able to reason the pdf of the inclinations by themselves follow a function of $\text{PDF}(i) = \frac{1}{4}|{sin(i)}|$, after all, it should start at zero, peak at π/2 and 3π/2, be zero at π, and integrate to 1.

But we want to identify some PDF function, $\text{PDF}(i(x))$, that is uniform across the hemisphere within small bins of dx. It turns out the transformation $i(x) = arccos(x)$ accomplishes exactly that. To see how let's look at the formula to transform the PDF from a variable, $i$, to $x$:

$\text{PDF}(i) di = \text{PDF}(i(x))| \frac{di}{dx}|dx = \text{g}(x)dx$

Where g(x) is our desired distribution function, I didn't write PDF because it's not necessarily normalized yet.

In our case $\text{PDF}(i(x)) = \frac{1}{4} sin(arccos(x))$ and $| \frac{di}{dx}| = \frac{1}{\sqrt{1 - x^2}}$ so the transformation is:

 $\text{PDF}(i) = \frac{1}{4} sin(arccos(x)) \frac{1}{\sqrt{1 - x^2}} = \frac{1}{4}dx$ Which is a constant! (I recommend graphing this in Desmos to see for yourself)

 So the function $g(x) = cos(i)$ should be uniform over the hemisphere.

 Credit to Keaton Bell at https://keatonb.github.io/archivers/uniforminclination for the help and figure.


In [17]:
# Generate the x axis for the cosine of the inclination
x_axis_cosI = np.linspace(-1, 1, 1000)

cosI_pdf = np.linspace(1/2,1/2,1000) # Creating our uniform distribution (PDF) for cos(I)
# And creating the CDF based on the PDF, with our function created earlier
cosI_cdf = CDF_generator(cosI_pdf, x_axis_cosI)

# Plotting the uniform CDF
plt.figure(figsize=(10, 8))
plt.plot(x_axis_cosI, cosI_cdf, color='blue', label='CDF for the cosine of the inclination')
plt.xlabel('cos(I)', fontsize=20)
plt.ylabel('CDF', fontsize=20)
plt.tick_params(direction='in', labelsize=16)
plt.xlim(-1, 1)
plt.legend(fontsize=16)
plt.title('Cumulative Density for cos(I)', fontsize=20)
plt.show()

In [18]:
# Now, use our previously generated functions to create the interpolated CDF
CDFcosI_interp = sci.interpolate.interp1d(x_axis_cosI, cosI_cdf)
# And generate a sample
sample_cosI = sample_generator(2500, CDFcosI_interp, x_axis_cosI)


# Plotting the generate samples, to sanity check that our generation makes sense
plt.figure(figsize = (10, 8))
plt.hist(sample_cosI, density = True, label = 'Generated cos(I) Sample', bins = 20)
plt.plot(x_axis_cosI, cosI_pdf, color = 'black', linestyle = '--', label  = 'Expected PDF generation for cos(I)')
plt.xlabel('cos(I)', fontsize = 20)
plt.ylabel('PDF', fontsize = 20)
plt.tick_params(direction = 'in', labelsize = 16)
plt.legend(fontsize = 16)
plt.xlim(-1, 1)
plt.show()

## Generating eccentricity \(e\)

Eccentricity is also not uniform. It is defined to be between 0 and 1, where an eccentricity of 0 represents a perfectly circular orbit and an eccentricity of 1 indicates a parabola, so as the eccentricity increases, the orbit strays further and further from a circle and becomes more eccentric.

For this, We used this figure from the 2010 research paper "A Survey of Stellar Families: Multiplicity of Solar\-Type Stars" by Raghavan et al. The dark, dashed line represents the 82 systems with $P \ge 1000$  days, and the lighter, dotted line represents the 35 systems with $P < 1000$  days \(but $P \ge 12$ days in both cases\). For orbits less than 12 days they find them to be highly circularized due to tidal interactions between the two stars. To represent this in our code stars with periods less than 12 days will have their eccentricities set to 0. For larger orbits their eccentricities will be pulled from one of the above PDFs depending on the exact value of their period.


In [19]:
# Getting these values from Raghavan, 2010 (Fig. 15)
ecc_less_1000_days_not_normal = [4/34, 5/34, 5/34, 6/34, 2/34, 7/34, 2/34, 1/34, 1/34, 1/34]
ecc_more_1000_days_not_normal = [3/81, 10/81, 9/81, 11/81, 11/81, 13/81, 8/81, 9/81, 3/81, 4/81]

#set x_space for eccentricity
x_axis_ecc = np.linspace(0, 1, 10)


#find normalization factor of the eccentricity
#equal to 1 over the integral of the histogram
normal = 1 / (sum(ecc_less_1000_days_not_normal) * (x_axis_ecc[1] - x_axis_ecc[0]))


#convert from list to array
ecc_less_1000_days = np.array(ecc_less_1000_days_not_normal)*normal
ecc_more_1000_days = np.array(ecc_more_1000_days_not_normal)*normal


#find and interpolate the CDFs for both the cases where the period is less than and greater than 1000 days
cdf_ecc_less_1000_days = CDF_generator(ecc_less_1000_days, x_axis_ecc)
cdf_ecc_less_interp = sci.interpolate.interp1d(x_axis_ecc, cdf_ecc_less_1000_days)

cdf_ecc_more_1000_days = CDF_generator(ecc_more_1000_days, x_axis_ecc)
cdf_ecc_more_interp = sci.interpolate.interp1d(x_axis_ecc, cdf_ecc_more_1000_days)

# Ordered from 0.0 to 1.0
period = 1005 #GET THE PERIOD HERE SOMEHOW!
if period <= 12:
    print("hello")
elif period <= 1000:
    # Create the CDF
    # And interpolate it so we can do the inverse random sampling
    
    sample_ecc = sample_generator(1000, cdf_ecc_less_interp, x_axis_ecc)
    
    plt.figure(figsize=(10, 8))
    plt.hist(sample_ecc, density = True, color='blue', label='CDF for the eccentricity')
    plt.plot(x_axis_ecc, ecc_less_1000_days)
else:
    # Create the CDF
    
    # And interpolate it so we can do the inverse random sampling
    
    sample_ecc = sample_generator(1000, cdf_ecc_more_interp, x_axis_ecc)
    
    plt.figure(figsize=(10, 8))
    plt.hist(sample_ecc, density = True, label='CDF for the eccentricity')
    plt.plot(x_axis_ecc, ecc_more_1000_days)
    #label axes
plt.show()

## Bonus: Radius Generation!

Luckily, the radii won't follow a random distribution becuase the radius depends solely on the mass of the star. So if we generate the mass of the star we'll determine the radius of the star. The relation between the mass and the radius follows the form:

$\frac{R}{R_{\odot}} = \frac{M}{M_{\odot}}^{\xi}$ where $\xi = 0.8$ for $M < 1 M_{\odot}$ and $\xi = 0.53$ for $M > 1 M_{\odot}$ 



In [20]:
#Generate a radius for the stars in solar radii

def radius_generator(M):

    #Check to see if the mass is large
    if M < 1:

        #Use larger exponent
        xi = 0.8

    #if star IS massive
    else:
        #Use the lower exponent
        xi = 0.53

    #Generate the radius in solar radii
    R = M ** xi

    return R
    

Now that we have figured how to generate samples for each individual parameter \(except for mass ratio but we'll incorporate that into the "Generating Binary Systems" function itself\-you'll see why\), we can swiftly move on to generating random binary star systems!


# Chapter 3: Generating A Random Sample of N Binary Systems

With the above distributions, we need to make a set of Binary systems to comprise our population. Of these, we will investigate which ones eclipse and eventually check the distributions of their masses, periods, eccentricities, and angular components to check how it stacks up against the initial population.

To make a large sample we'll make one function with one input, N, which dictates how many systems will be generated.  The function will then go through and generate N random values from each distribution for this sample. We want to make something that's fairly efficient but keeps in mind the complex relationships between the parameters at play. For example, the period of the orbits do not depend on the masses or orbital angles of the orbit and so N random periods can be generated at once. This saves time as we only need to build the CDF from the Salpeter PDF one time and just get N numbers we can store for later. However, the eccentricities DO depend on the period of the orbits. This means we'll need to generate our periods first and use that information as a prior to what eccentricities are more likely. We also want to keep the efficient step of only building our CDFs once to reduce computational load.

Another pitfall we want to avoid is with the stellar masses. Although the primary mass can be generated from the Salpeter IMF without issue the secondary mass is not as simple. If we just generate a secondary mass from uniform distributions of mass ratios from 0.1 to 1 then we may end up generating impossibly small stars around less massive primary stars. To prevent this we need to carefully set the lower bound of the mass ratio, q, to be free to change in case the mass of the primary star is sufficiently low. This is to avoid generating a secondary star with a mass of $0.03 M_{\odot} \leq M_{2} \leq 0.1$ where $M_2$ is the secondary mass. Such a condition is given by:

$q \in [\text{min}(0.1, \frac{0.1}{M_1}), 1]$

We can see for primary masses less than one solar mass then the minimum mass ratio will rise to prevent very small masses from being generated.

After all the values are generated we'll have a few independent lists. To bring them all together we simply need to call np.array which will take our lists and paste them together as columns in a 2D array. Each row will represent one system and each column will represent a parameter such as Mass.


In [21]:
#Enter integer
#Return 2D array of binary systems
#with rows of primary mass (M_1), secondary mass (M_2), primary radius (R_1), secondary radius (R_2)
#log P, eccentricity, ω, Ω, cos(i)
def pop_genesis(N):

    ##First, generate the Masses and Radii
    #Generate the first mass using the Salpeter IMF
    #Then generate the second mass from the mass ratio
    #use the masses to calculate the radii in solar radii
    #using radius_generator()
    
          
    ##Radii will be necessary for when we need to calculate
    #Projected separation later on
    
    #generate list of masses of primary star in solar masses
    
    M_1 = sample_generator(N, Salpeter_cdf_interp, M)
    
    #initialize lists to hold secondary mass
    #radius of primary star
    #and radius of secondary star
    M_2 = []
    R_1 = []
    R_2 = []

    for mass_1 in M_1:
        
        #set lower bound of mass ratio to prevent undermassive stars
        lower_bound_q = max(0.1/mass_1, 0.1)
        
        #generate random value of q
        q = np.random.uniform(lower_bound_q, 1)
        
        #calculate mass of secondary star in solar mass using mass ratio
        mass_2 = mass_1 * q
        
        #add them to the list
        M_2.append(mass_2)
        
        # Also, generate the radii in this same loop in solar radii
        r1 = radius_generator(mass_1)
        r2 = radius_generator(mass_2)
        # And add them to the radii lists
        R_1.append(r1)
        R_2.append(r2)
 
    
    
    ##Third, generate Period and Eccentricity of the orbit
    ##The Period is pulled from a log-normal distribution
    ##which is then used to determine from what distribution
    ##from Raghavan we'll used
    
    #Generate list of periods in days in log space
    log_P_sample = sample_generator(N, log_P_cdf_interp, log_P)
    
    Ecc = []
    #Generate eccentricity based on the period of the binary
    for log_p in log_P_sample:
        # Calculate the period based on log_p
        period = 10**log_p
        # Now, check: if the period is really small, the eccentricity should be set to 0
        if period <= 12:
            e = 0
        # If its between 12 and 1000, it should follow the CDF calculated before
        elif period <= 1000:
            e = sample_generator(1, cdf_ecc_less_interp, x_axis_ecc)[0]
        # And if its more than 1000, do the same as we found beforehand
        else:
            e = sample_generator(1, cdf_ecc_more_interp, x_axis_ecc)[0]
        Ecc.append(e)
        

    
    ##Fourth, generate all the angular components, Ω, ω, cos(i)
    ##of all the orbits

    Omega = np.random.uniform(0, 2 * np.pi, N) #Ω
    omega = np.random.uniform(0, 2 * np.pi, N) #ω
    cos_i = np.random.uniform(-1, 1, N) #cos(i)
    
    
    ##Finally, take all these lists and paste them together into one
    ##Array
    
    binary_population = np.column_stack((M_1, M_2, R_1, R_2, log_P_sample, Ecc, Omega, omega, cos_i))
    
    return binary_population

population = pop_genesis(1000)
print(population[0])

[0.13026733 0.11884437 0.19582513 0.18196292 6.41673096 0.26831055
 4.92543347 3.16998194 0.23344764]


# Chapter 4: The Fun Part–Detecting Eclipses

Now comes for the key portion of the project. How do we know when a binary is eclipsing?

If we take the X axis as the reference angle for an observer they will only notice the motion in the Y-Z plane. We will need to calculate and log the Y-Z coordinates for our binaries as a function of theta as θ runs from 0 to 2π. Those coordinates depends on the angles, semi-major axis (a), and eccentricity such that:

$y = r[\text{sin}(\Omega) *\text{cos}(\theta) + \text{cos}(\Omega)\text{sin}(\theta)\text{cos}(i)]$

$z = r[\text{sin}(\theta)\text{sin}(i)]$

where r is given by:

$r = a \frac{(1 - e^2)}{1 + e\text{cos}(\theta - \omega)}$

a is the semi-major axis and can be gained from the period and the masses of the stars using Kepler's 3rd law and Newton's Law of gravity:

$a^3 = \frac{G(M_1 + M_2)}{4\pi^2} T^2$

$a = (\frac{G(M_1 + M_2)}{4\pi^2} T^2)^{1/3}$

**We need to be careful with the units here**

With these coordinates we can tell if the stars are eclipsing if the distance between their centers is less than the sum of their radii. If we set the primary star at the origin and say it's stationary compared to the secondary star then we will get an eclipse if:

$\sqrt{y(\theta)^2 + z(\theta)^2} < R_1 + R_2$

where y and z are function of theta. We can call the left hand side of the equation the scaled separation distance, such that

$s(\theta) = \sqrt{y(\theta)^2 + z(\theta)^2}$


In [22]:
# distance as a function of theta (not the actual thing i just wanted to see how I can calculate it)
num_steps = 360
theta = np.linspace(0,2*np.pi,num_steps) #in radians

# calculate semi-major axis (a) in metres as a function of M_1 and P
def semimajoraxis(M_1, M_2, log_P_sample):
    # convert log_P back to P first, and subsequently convert to seconds for Kepler's Third Law calculations
    
    days_to_secs = 86400
    
    P = (10**log_P_sample) * days_to_secs
    
    # similarly, convert M_1 and M_2 from solar masses to kg
    
    sol_mass_to_kg = 1.989e+30
    
    M1 = M_1 * sol_mass_to_kg
    M2 = M_2 * sol_mass_to_kg
    
    # apply kepler's third law to find the semi-major axis in terms of M_1 and P: (P^2 = (4п^2 / (G * (M_1+M_2)) * a^3), so
    # first define Gravitational Constant G
    
    G = 6.674*10**-11 #m^3 kg^-1 s^-2
    
    # a in meters   
    a = np.cbrt((P**2 * G * (M1+M2)/(4 * (np.pi)**2)))
    return a

# using the data from population[0]
M_1 = population[0][0]
M_2 = population[0][1]
log_P_sample = population[0][4]
Ecc = population[0][5]
Omega = population[0][6]
omega = population[0][7]
cos_i = population[0][8]
  
    
def calc_pos(theta, M_1, M_2, log_P_sample, Ecc, omega, Omega, cos_i):
    #Generate emi-major axis (in meters) using Kepler's 3rd Law
    a = semimajoraxis(M_1, M_2, log_P_sample)
    #Convert to AU
    
    m_to_AU = 6.684*10**-12
    
    a = a * m_to_AU
    # the line below is Kepler's 1st law because it shows that the relative distance between the centres of each star, r, follows an ellipse
    r = a*(1-Ecc**2)/(1+Ecc*np.cos(theta-omega)) # the relative distance between the centres of each star in AU
    
    # Calculate the 3d positions as a function of theta
    x = r*(np.cos(Omega)*np.cos(theta) - np.sin(Omega)*np.sin(theta) * (cos_i))
    y = r*(np.sin(Omega)*np.cos(theta) + np.cos(Omega)*np.sin(theta) * (cos_i))
    z = r*np.sin(theta)*(1-(cos_i)**2)**0.5 #Keep in mind we may want negative values of sin(i)
    return x, y, z

## Scaled Separation Distance

We can tell if the stars are eclipsing if the apparent distance between their centers is less than the sum of their radii. If we set the primary star at the origin and say it's stationary compared to the secondary star then we will get an eclipse if:

$\sqrt{y(\theta)^2 + z(\theta)^2} < R_1 + R_2$

where y and z are function of theta. We can call the left hand side of the equation the scaled separation distance, such that

$s(\theta) = \sqrt{y(\theta)^2 + z(\theta)^2}$

Recall that in this work we are looking along the $\hat{X}$ axis, consistent with the figure at the beginning of the notebook. This means all motion will be seen in the YZ plane.

One thing to note is that we are not accounting for the change in apparent size of the secondary star as it moves around the primary star. In theory secondary eclipses should be harder because the secondary star is smaller in apparent size. However, in practice the sizes involved are such that neither star changes in apparent size as they move. The secondary orbits are on the scale of around a few hundred AU maximum but the distances from Earth are tens or hundreds of thousands of AU so the secondary star appears about the same size throughout it's journey. This means that we can say that if the stars are closer than the sum of the radii then they will appear to overlap and eclipse without worrying about how the angular size of the secondary star changes. A quick back of the envelope calculation will show that even in the case of large orbits (~100 AU) and very close systems (about 2.5 light years, closer than the nearest non-solar star) not accounting for the change in angular size of the secondary star will only make a 0.06% error on your estimation of the true size of the secondary.


In [42]:
def separation_dist(theta, M_1, M_2, log_P_sample, Ecc, omega, Omega, cos_i):
    #calculate orbit for binary system
    #coords in AU
    
    x, y, z = calc_pos(theta, M_1, M_2, log_P_sample, Ecc, omega, Omega, cos_i)
    
    #calculate separation distance
    #to re-iterate, we're viewing along the X axis, like the figure at the beginning of
    #the notebook, so the apparent motion will lie in the YZ plane
    
    s = (y**2 + z**2)**0.5 #in AU
    
    return s

population = pop_genesis(1)

# using the data from population[0]
M_1 = population[0][0]
M_2 = population[0][1]
log_P_sample = population[0][4]
Ecc = population[0][5]
Omega = population[0][6]
omega = population[0][7]
cos_i = population[0][8]

positions = calc_pos(theta, M_1, M_2, log_P_sample, Ecc, omega, Omega, cos_i)

x = positions[0] # x over the orbit
y = positions[1] # y over the orbit
z = positions[2] # z over the orbit

# projected separation using function

s = separation_dist(theta, M_1, M_2, log_P_sample, Ecc, omega, Omega, cos_i)

# convert star radii from solar radii to AU

solar_r_to_m = 6.957e8

solar_r_to_au = 0.00465

R_1 = population[0][2] * solar_r_to_au

R_2 = population[0][3] * solar_r_to_au
    
# Plot the scaled projected separation
plt.scatter(np.degrees(theta),s, c = theta, s=2, cmap = 'cool')
plt.axhline(y= R_1 + R_2 , color='black', linestyle='--')
plt.xlabel('theta (deg)',fontsize=18)
plt.ylabel('s (AU)',fontsize=18) #add units to y axis
plt.tick_params(direction = 'in', labelsize = 16)
plt.xticks([0,45,90,135,180,225,270,315,360])
plt.xlim([0,360])
plt.show()

#plot the orbits of the secondary star
m_to_AU = 6.684*10**-12
plt.scatter(y, z, c = theta, cmap='cool', s=10)
plt.scatter(0,0, color = 'red', label = 'primary star', s = 50)
plt.xlabel('y coordinate (AU)', fontsize = 18)
plt.ylabel('z coordinate (AU)', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 16)
plt.legend(fontsize = 16)
plt.show()

What incredible\-looking orbits! Keep in mind that if the scaled separation distance drops anywhere below the horizontal dashed line, which is defined as the sum of the two radii of the two stars \(just scaled down!\), an eclipse will occur. Now that we have defined the conditions for which an eclipse will happen, we can create a function that helps us find eclipses from a random sample of N binary star systems!


In [24]:
def eclipse_finder(N):
    
    #Generate a popuation for binaries
    
    binary_sample = pop_genesis(N)
    
    #define theta angle
    
    theta = np.linspace(0,2*np.pi,num_steps)
    
    #initialize boolean function to tell which systems are eclipsing
    
    eclipse_happens = []
    
    for ii in range(N):
        
        #initialize boolean value as false, will change to true
        #if an eclipse actually happens
        eclipse = False
        
        #unpack parameters of the binary system ii
    
        M_1 = binary_sample[ii, 0] #in solar mass
    
        M_2 = binary_sample[ii, 1] #in solar mass
    
        R_1 = binary_sample[ii, 2] #in solar rads
    
        R_2 = binary_sample[ii, 3] #in solar rads
        
        #convert radii to AU
        
        solar_r_to_au = 0.00465
        
        R_1 = R_1 * solar_r_to_au
        
        R_2 = R_2 * solar_r_to_au
    
        log_P_sample = binary_sample[ii, 4] #in days
    
        Ecc = binary_sample[ii, 5]
    
        omega = binary_sample[ii, 6] #in radians
    
        Omega = binary_sample[ii, 7] #in radians
    
        cos_i = binary_sample[ii, 8]
        
        #calculate separation distances as a function of theta for this binary system
        
        separation = separation_dist(theta, M_1, M_2, log_P_sample, Ecc, omega, Omega, cos_i)
        
        #pull out minimum value of the separation
        #if it's less than the sum of the radii then we have an eclipse
        if min(separation) < (R_1 + R_2):
            eclipse = True
        
        eclipse_happens.append(eclipse)
    
    return binary_sample[eclipse_happens], binary_sample

In [25]:
N = 100000

eclipsing_population, general_population = eclipse_finder(N)

print('The rate of eclipsing among binaries is ' + str(len(eclipsing_population)/len(general_population)))
print('We got a total of ' + str(len(eclipsing_population)) + ' eclipsing binaries out of ' + str(N) + ' generated binaries')

The rate of eclipsing among binaries is 0.00872
We got a total of 872 eclipsing binaries out of 100000 generated binaries


## Accounting for uncertainties

If there's one thing we should have already picked up from this class it's that there's no such thing as absolutely precise measurement. There's always an error or uncertainty we need to account for. No graph should be lacking in error bars because otherwise, we cannot be sure whether or not the differences in two outcomes is statistically significant.

In our case we will have two populations, one that is eclipsing and one that is for the general population. We will then bin the data and plot their distribtion. The main source of uncertainty is in the number counts of items in each bin. For this we can use Poisson Error to model the error.

So imagine taking some histogram:

Now focus on one bin of that histogram. That bar is counting up the number of values that falls between two values. But that number is not constant, different trials may return different values for that single bin. It turns our the number of items in that bin will also follow a distribution! What are some things we can say about this distribution? In our case we can say for certain the events (whether or not some objects are included in this bin) are independent. This means that the value of some particular parameter such as eccentricity in one binary stystem won't affect the value of the next. We can also say that for our populations, both the general population and the binary population, the rate at which some value such as eccentricity falls into some bin is independent of the number of objects in the population. Regardless of whether or not we're talking about a population of 100 binaries or 1,000,000 binaries, only a set fraction of them will fall into a certain bin.

With these two conditions adequately met we can model the distribution of number of items in each bin of the histogram as a Poisson distribution! For our purposes the success of a trial corresponds to an object belonging to our bin in question. So if we have a population of size n and the probability that any one of those objects falls into our bin is p then we get the factor λ as $\lambda = np$. But notice $\lambda = np \approx N$ where N is the number of items we counted in the bin! Since the variance of a Poisson distribtution is $\sigma^2 = \lambda$ the uncertainty can be expresses as $\sigma = \sqrt{N}$.

That may have felt like a long walk for a small drink of water but what we've done is pretty powerful. We've found we can use our histogram to estimate the uncertainty with a really easy form! So long as we make sure that the events are independent and the rate of success doesn't depend on the size of the sample this should work. In astro terms these are pretty easily met as it's easy to assume most times that just becuase one binary has a certain value of eccentricity or period of orbit won't affect another, so they should be independent. We also can assume the size of our survey or generated random sample shouldn't affect the rate at which we find binaries to have a certain eccentricity or period, it's an inherent quality of the binary system. This, of course, isn't a silver bullet. It's debated whether or not Poisson is overused in the field and whether or not binomial confidence interval (deriving from a more general binomial distribution) is more reliable. So always make sure to do your homework before choosing an error analysis, especially in the case of small number counts in each bin.

Below is a code to generate Poisson error for any normalized histogram. It's pretty straightforward, we'll generate a histogram of values using np.histogram() and take the square root of the resulting array, which should return an array of the uncertainty of each bin of the histogram. Then we need to normalize it by dividing the error by the integral of the histogram of the data, equal to the sum of the values of the histogram times the bin width.

For ease of future graphing we'll also have the function return the x-coordinates of the bin centers and the y corrdinates of the normalized histograms. This is simply because we know that in the next cell we'll be calling plt.errorbar() which needs these two things as callables so it'd be nice to offload that work to this function as well.


In [26]:
#number of bins desired
num_bins = 21


#define bins for all our desired parameters

M_bins = np.linspace(0.1, 3, num_bins)

q_bins = np.linspace(0.1, 1, num_bins)

log_P_bins = np.linspace(-2, 12, num_bins)

ecc_bins = np.linspace(0, 1, num_bins)

omega_bins = np.linspace(0, 2*np.pi, num_bins)

Omega_bins = np.linspace(0, 2* np.pi, num_bins)

cos_i_bins = np.linspace(-1, 1, num_bins)



def normalized_poisson_error(data, bins):

    #create the histogram of the array

    data_hist, bins = np.histogram(data, bins = bins)
    

    #find the poisson uncertainty

    poisson_error = abs(data_hist)**0.5
  

    #normalize it by dividing by the integral of the histogram

    poisson_error_normalized = poisson_error / (sum(data_hist) * (abs(bins[1] - bins[0])))
    
    #find the y values of the normalized distribution so we can plot it
    
    data_hist_norm, bins = np.histogram(data, bins = bins, density = True)

    #find bin centers

    #while loop works best because the bin centers list won't be the same size

    bin_centers = []

    i = 0

    while i < len(bins) - 1:

        #calulate the average of two bin values and add it to the list
        bin_centers.append((bins[i] + bins[i+1])/2)

        i = i + 1

    return bin_centers, data_hist_norm, abs(poisson_error_normalized)

Now let's plot the distributions for each individual parameter for both our binary population and eclipsing population and compare them! This is how we will do it:


In [27]:
##Plotting primary stellar mass

eclipsing_mass = eclipsing_population[:, 0]

mass_bin_centers, eclipsing_mass_norm, eclipsing_mass_sigma = normalized_poisson_error(eclipsing_mass, M_bins)


general_mass = general_population[:, 0]

mass_bin_centers, general_mass_norm, general_mass_sigma = normalized_poisson_error(general_mass, M_bins)


plt.hist(eclipsing_mass, bins = M_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Eclipsing Population')

plt.errorbar(mass_bin_centers, eclipsing_mass_norm, yerr = eclipsing_mass_sigma,
             linestyle = '', capsize = 4, color = 'red', alpha = 0.5)

plt.hist(general_mass, bins =  M_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'General Population')

plt.errorbar(mass_bin_centers, general_mass_norm, yerr = general_mass_sigma,
             linestyle = '', capsize = 4, color = 'blue')

plt.xlabel(r'Primary Mass ($M_{\odot}$)', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()



#Plotting mass ratio q


eclipsing_q = eclipsing_population[:, 1]/eclipsing_population[:, 0]

q_bin_centers, eclipsing_q_norm, eclipsing_q_sigma = normalized_poisson_error(eclipsing_q, q_bins)


general_q = general_population[:, 1]/general_population[:, 0]

q_bin_centers, general_q_norm, general_q_sigma = normalized_poisson_error(general_q, q_bins)



plt.hist(eclipsing_q, bins = q_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Eclipsing Population')

plt.errorbar(q_bin_centers, eclipsing_q_norm, yerr = eclipsing_q_sigma,
             linestyle = '', capsize = 4, color = 'red', alpha = 0.5)

plt.hist(general_q, bins =  q_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'General Population')

plt.errorbar(q_bin_centers, general_q_norm, yerr = general_q_sigma,
             linestyle = '', capsize = 4, color = 'blue', alpha = 0.5)

plt.xlabel('Mass Ratio q', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()


#Plotting period of orbit

eclipsing_log_p = eclipsing_population[:, 4]

log_p_bin_centers, eclipsing_log_p_norm, eclipsing_log_p_sigma = normalized_poisson_error(eclipsing_log_p, log_P_bins)


general_log_p = general_population[:, 4]

log_p_bin_centers, general_log_p_norm,  general_log_p_sigma = normalized_poisson_error(general_log_p, log_P_bins)


plt.hist(eclipsing_log_p, bins = log_P_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Eclipsing Population')

plt.errorbar(log_p_bin_centers, eclipsing_log_p_norm, yerr = eclipsing_log_p_sigma,
             linestyle = '', capsize = 4, color = 'red', alpha = 0.5)
         
plt.hist(general_log_p, bins =  log_P_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'General Population')

plt.errorbar(log_p_bin_centers, general_log_p_norm,  yerr = general_log_p_sigma,
             linestyle = '', capsize = 4, color = 'blue', alpha = 0.5)

plt.xlabel(r'log(P [Days])', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()


##Plotting eccentricity

eclipsing_ecc = eclipsing_population[:, 5]

ecc_bin_centers, eclipsing_ecc_norm, eclipsing_ecc_sigma = normalized_poisson_error(eclipsing_ecc, ecc_bins)



general_ecc = general_population[:, 5]

ecc_bin_centers, general_ecc_norm, general_ecc_sigma = normalized_poisson_error(general_ecc, ecc_bins)


plt.hist(eclipsing_population[:, 5], bins = ecc_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Eclipsing Population')

plt.errorbar(ecc_bin_centers, eclipsing_ecc_norm, yerr = eclipsing_ecc_sigma,
             linestyle = '', capsize = 4, color = 'red', alpha = 0.5)
         
plt.hist(general_population[:, 5], bins =  ecc_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'General Population')


plt.errorbar(ecc_bin_centers, general_ecc_norm, yerr = general_ecc_sigma,
             linestyle = '', capsize = 4, color = 'blue', alpha = 0.5)


plt.xlabel(r'Eccentricity', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()


##Plotting ω

eclipsing_omega = eclipsing_population[:, 6]

omega_bin_centers, eclipsing_omega_norm, eclipsing_omega_sigma = normalized_poisson_error(eclipsing_omega, omega_bins)


general_omega = general_population[:, 6]

omega_bin_centers, general_omega_norm, general_omega_sigma = normalized_poisson_error(general_omega, omega_bins)


plt.hist(eclipsing_population[:, 6], bins = omega_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Eclipsing Population')


plt.errorbar(omega_bin_centers, eclipsing_omega_norm, yerr = eclipsing_omega_sigma,
             linestyle = '', capsize = 4, color = 'red', alpha = 0.5)
         
plt.hist(general_population[:, 6], bins = omega_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'General Population')

plt.errorbar(omega_bin_centers, general_omega_norm, yerr = general_omega_sigma,
             linestyle = '', capsize = 4, color = 'blue', alpha = 0.5)

plt.xlabel(r'ω', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()


##Plotting Ω

eclipsing_Omega = eclipsing_population[:, 7]

Omega_bin_centers, eclipsing_Omega_norm, eclipsing_Omega_sigma = normalized_poisson_error(eclipsing_Omega, Omega_bins)


general_Omega = general_population[:, 7]

Omega_bin_centers, general_Omega_norm, general_Omega_sigma = normalized_poisson_error(general_Omega, Omega_bins)



plt.hist(eclipsing_population[:, 7], bins = Omega_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Eclipsing Population')

plt.errorbar(Omega_bin_centers, eclipsing_Omega_norm, yerr = eclipsing_Omega_sigma,
             linestyle = '', capsize = 4, color = 'red', alpha = 0.5)
         
plt.hist(general_population[:, 7], bins =  Omega_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'General Population')

plt.errorbar(Omega_bin_centers, general_Omega_norm, yerr = general_Omega_sigma,
             linestyle = '', capsize = 4, color = 'blue', alpha = 0.5)

plt.xlabel(r'Ω', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()


##Plotting cos(i)

eclipsing_cos_i = eclipsing_population[:, 8]

cos_i_bin_centers,  eclipsing_cos_i_norm, eclipsing_cos_i_sigma = normalized_poisson_error(eclipsing_cos_i, cos_i_bins)


general_cos_i = general_population[:, 8]

cos_i_bin_centers, general_cos_i_norm, general_cos_i_sigma = normalized_poisson_error(general_cos_i, cos_i_bins)



plt.hist(eclipsing_population[:, 8], bins = cos_i_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Eclipsing Population')

plt.errorbar(cos_i_bin_centers,  eclipsing_cos_i_norm, yerr = eclipsing_cos_i_sigma,
             linestyle = '', capsize = 4, color = 'red', alpha = 0.5)
         
plt.hist(general_population[:, 8], bins =  cos_i_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'General Population')


plt.errorbar(cos_i_bin_centers, general_cos_i_norm, yerr = general_cos_i_sigma,
             linestyle = '', capsize = 4, color = 'blue', alpha = 0.5)

plt.xlabel(r'cos(i)', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()

## Investigation: What do these histograms tell us about the eclipsing population?

Let's look at the parameters that differ from the original binary systems population. Primary Masses, Mass Ratios q and ω of the eclipsing population do not differ much from our binary population.

### Period

There is a big difference in periods between the eclipsing population and the binary systems population. Eclipsing systems tend to have a shorter period, and this makes sense. According to Kepler's Third Law, $a^3\propto T^2$: if period increases then so does the semi\-major axis, making the orbit VERY big, and thus, the now relatively tiny stars in the system would have a smaller chance of eclipsing.

### Ω

Ω describes the angle between the reference direction and the point at which the orbit intersects the reference plane. The three peaks seen in the PDF of Ω are respectively 0, π, and 2π, but why is this the case? When the orbit of the secondary star intersects the reference plane at 0 degrees–or in other words, right in front of us–then it will have a greater chance of being right between our line of sight and the primary star, forming a primary eclipse! 0 and 2π are the exact same, and π is just the same thing on the other side of the primary star, thus forming a secondary eclipse.

It is also worth noting that for our model, Ω affects the rate of eclipses even though Ω doesn't affect anything in David's notebook! This is because our reference direction is from the x\-axis, while David's is from the z\-axis, thus the parameter that doesn't affect eclipses is the other angular parameter ω for us.

### Angle of Inclination \(I\)

On our graph, we can see that eclipsing systems mainly have an inclination of either arccos\(\-1\) or arccos\(1\), which both give the same values of 0, π, and 2π. All 3 of these values mean the same thing: the orbit basically lies on the reference plane. This makes sense because if it does lie on our reference plane, there is a higher chance of the secondary star and the primary star aligning with our reference direction and thus forming an eclipse.

### Eccentricity

What we found most interesting is how different the eccentricity of the eclipsing population appears when compared to the binary systems population. It shows that significantly more eclipses happen when eccentricity is close to 0 and close to 1. How can this be the case? It's not like the angular parameters where 0 is the same as $2\pi$...an eccentricity of 0 and 1 are completely different orbits: zero is perfectly circular, and one is essentially a parabola.

For the eccentric orbits that have eclipses, we can explain this pattern geometrically. Say we have a very eccentric orbit, for example, one that is a very stretched\-out ellipse. If we rotate it along its semi\-major axis, the projection of the semi\-minor axis would not change much compared to a less eccentric orbit. This means that for a larger range of angles of inclination \(I\), from our reference direction, the semi\-minor axis would still appear relatively close to the origin for an eclipse to occur.

For the mostly circular orbits that have eclipses, we can hypothesize an explanation for this using the non\-uniform distribution of periods and eccentricity. For binary systems with periods &lt; 12 days, we set the distribution of its eccentricity to be between 0 and 0.05, making its orbit very round.
For these systems with very short periods, since once again, according to Kepler's Third Law, $a^3\propto T^2$, so their semi\-major axes would also be very small, meaning that the two stars are close together and thus have a higher chance of eclipsing. We can further investigate this by creating a PDF of the eccentricities of binary systems with periods &gt; 12 days:

### Eclipsing Rate

Our eclipsing rate of $\lesssim 1\%$ is smaller than most other groups, who seem to converge around $1.5\%$. It's important to know the calculated eclipsing rate will be sensitive to the PDFs passed to create the populations. Of particular importance will be the mass, period, and eccentricity distributions (aka the only non uniform distributions applied in this work). Most every group used the recommended Salpeter IMF but other choices will yeild different results:

As we can see above the Salpeter IMF is going to create many more low mass stars in comparison to high mass stars. For the purposes of this study non-Salpeter IMFs should lead to a higher eclipsing rate as generating more massive stars means generating larger stars which aids in how often things eclipse in our simulation.

Periods seem like they should have identical results for all groups given we all had to work with a log-normal distribution. However, the key detail left to the discretion of the students are the bounds of the distribution. If they're set to be very wide (say, out to 3σ - 5σ) the very low period orbits (that may represent close contact binaries or unphysically close systems if set too low) will be detected as eclipsing basically 100% of the time. However, more restrictive bounds (of 2σ) will not have as many close orbits that are commonly detected as a eclipsing.

Relevant to the results of this project is the generation of the eccentricities, which we believe to be the source of the lower occurrence rate of eclipsing binaries. Recall our pdfs for eccentricities for orbits greater than 12 days are adopted directly from Raghavan 2010:

One thing to note from this distribution is the low rate of occurrence of high eccentricity orbits. We confirm below that high eccentricity orbits are heavily preferred in detecting binary systems with large orbits. Other groups use other methods such as using a uniform distribution between 0 and 1 for any orbit greater than 12 days or a uniform distribution between bounds determined by the period to generate eccentricities. Both of the methods are more likely to generate a high eccentricity orbit than ours and so their eclipsing binary system will be more represented by large orbit, high eccentricity systems than ours. It's key to stress that this is not a reflection on the legitamacy of the models applied by each group. Each is reasonable and arguable given the limited data presented. The key takeaway from this is that the results of simulated physics hinges on certain assumptions made about underlying characteristics of the system that can be discrepant if the current state of knowledge is poor. It's reasons like these that show why stating systematic effects of assumptions and models is incredibly important in any astrophysical work.


In [28]:
# visualize eclipsing population with period > 12 days.

# create arrays that single out all the periods and eccentricities of the eclipsing population
log_P_ecl = eclipsing_population[:, 4]
ecc_ecl = eclipsing_population[:, 5]

# set log_P_long to be all the eclipsing systems which have a period of more than 12 days.
log_P_long = log_P_ecl > np.log10(12)

# create ecc_P_long, which stores all the eccentricities for eclipsing systems which have a period of more than 12 days.
ecc_P_long = ecc_ecl[log_P_long]

# plot our results
plt.hist(ecc_P_long, bins = 20, density = True, color = 'red')
plt.xlabel(r'Eccentricity', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.xlim(0,1)
plt.show()

This verifies our hypothesis! By isolating the binary systems that have a period of more than 12 days, the eclipsing population with eccentricities of near zero is almost entirely gone! This means that if we want an eclipse to happen, either we have a system with **very short period** \(&lt;12 days\) and hence a **circular orbit**, or we have a system with a very eccentric orbit so it eclipses even with a relatively large angle of inclination.



# Chapter 6: Comparing Primary Eclipses and Secondary Eclipses

Now, what if we wanted to see if there is a difference between the binary systems that have only one eclipse or both? Well, we need a new and improved version of the eclipse finder, the <span style='font-size:large'>**ECLIPSINATOR\_2000**</span><span style='font-size:large'>!</span>

The eclipsinator does a similar thing to the eclipse finder, except now we log at what exact values of θ these eclipses occur. If the values of theta are close to 0 or 2π then they must be closer to the observer and thus primary eclipses. Eclipses with theta values near π occur with the primary star between the viewer and the secondary and thus are secondary eclipses. The way we can keep track of the thetas is by utilizing the fact that all values of the apparent separation, s, has a corresponding θ value from the array variable theta. If we find the indices at which an eclipse occurred in the separation distances those will correspond to the indices of the theta array at which those eclipses occur. For example, if an eclipse occurs at s[20] then theta[20] will tell us at what angle that eclipse happened.

We then want to take this information and classify our eclipsing binaries into one of three groups: those that only have a primary eclipse, those that have only a secondary eclipse, and those that have both. The only trick here is that our code below will track if any binary system has a primary or secondary eclipse. If both are found, then we'll mark it accordingly.

In [29]:
#Create a function which helps us separate out the primary ecliipses and secondary eclipses within our eclipse population
def eclipsinator_2000(N):
    
    #Generate a popuation for binaries
    
    binary_sample = pop_genesis(N)
    
    #define theta angle
    
    theta = np.linspace(0,2*np.pi,num_steps)
    
    #initialize boolean function to tell which systems are eclipsing at all
    
    both_happens = []
    
    #initialize boolean function to tell which systems have a primary eclipse (agnostic to whether there's a secondary)
    
    primary_happens = []
    
    #initialize boolean function to tell which systems have a secondary eclipse (agnostic to whether there's a primary)
    
    secondary_happens = []
    
    for ii in range(N):
        
        #initialize boolean values as false, will change to true
        #if an both, only primary, only secondary happens
        
        both = False
        
        primary = False
        
        secondary = False
        
        #unpack parameters of the binary system ii
    
        M_1 = binary_sample[ii, 0] #in solar mass
    
        M_2 = binary_sample[ii, 1] #in solar mass
    
        R_1 = binary_sample[ii, 2] #in solar rads
    
        R_2 = binary_sample[ii, 3] #in solar rads
        
        #convert radii to AU
        
        solar_r_to_au = 0.00465
        
        R_1 = R_1 * solar_r_to_au
        
        R_2 = R_2 * solar_r_to_au
    
        log_P_sample = binary_sample[ii, 4] #in days
    
        Ecc = binary_sample[ii, 5]
    
        omega = binary_sample[ii, 6] #in radians
    
        Omega = binary_sample[ii, 7] #in radians
    
        cos_i = binary_sample[ii, 8]
        
        #calculate separation distances as a function of theta for this binary system
        
        separation = separation_dist(theta, M_1, M_2, log_P_sample, Ecc, omega, Omega, cos_i)
        
        
        #find all the systems that eclipse at all
        
        eclipsing = np.where((separation < R_1 + R_2))[0]
        
        #find at which angles, θ, an eclipse is happening
        
        eclipsing_theta = theta[eclipsing]
        
        #find where these angles correspond to primary eclipses
        
        primary_theta = any(np.append(eclipsing_theta < np.pi / 2, eclipsing_theta > 3*np.pi / 2))
        
        #find where these angles correspond to secondary eclipses
        
        secondary_theta = np.where((eclipsing_theta > np.pi / 2) & (eclipsing_theta < 3*np.pi/2))[0]
            
        
        #only if the condition for a primary eclipse is met at least once the list won't be empty
        #Thus, a primary eclipse happens (and an eclipse overall as well)
        if primary_theta == True:
            primary = True
        
        #only if the condition for a secondary eclipse is met at least once the list won't be empty
        #Thus, a secondary eclipse happens (and an eclipse overall as well)
        if len(secondary_theta) > 0:
            secondary = True
        
        #in the case both eclipses happened overwrite the primary and secondary only conditions
        #to be false and set both condtion to be true
            
        if primary == True & secondary == True:
            both = True
            primary = False
            secondary = False
            
            
        #append boolean values to our lists
                
        both_happens.append(both)
        
        primary_happens.append(primary)
        
        secondary_happens.append(secondary)
    
    return binary_sample, binary_sample[both_happens], binary_sample[primary_happens], binary_sample[secondary_happens]

In [30]:
# generate N binary systems
N = 100000

general_population, dual_eclipsing_population, primary_population, secondary_population = eclipsinator_2000(N)

print('The rate of dual eclipsing binaries is ' + str(len(dual_eclipsing_population)/len(general_population)) + ', with ' + str(len(dual_eclipsing_population)) + ' out of ' + str(len(general_population)))

print('The rate of primary only eclipsing binaries is ' + str(len(primary_population)/len(general_population)) + ', with ' + str(len(primary_population)) + ' out of ' + str(len(general_population)))

print('The rate of secondary only eclipsing binaries is ' + str(len(secondary_population)/len(general_population)) + ', with ' + str(len(secondary_population)) + ' out of ' + str(len(general_population)))

The rate of dual eclipsing binaries is 0.00619, with 619 out of 100000
The rate of primary only eclipsing binaries is 0.00146, with 146 out of 100000
The rate of secondary only eclipsing binaries is 0.00108, with 108 out of 100000


The rate of primary eclipses is not the same as the rate of secondary eclipses! Within our eclipsing population, there are more secondary eclipses than primary eclipses, meaning that the fact that a secondary eclipse occurs does not guarantee that a primary eclipse will also happen. This means that there are actually three types of systems with eclipses: ones with both a primary and a secondary eclipse, ones with only a primary eclipse and ones with only a secondary eclipse. We can single these out and plot them all on the same graph and analyze the differences:


In [31]:
##Plotting primary stellar mass

dual_eclipsing_mass = dual_eclipsing_population[:, 0]
mass_bin_centers, dual_eclipsing_mass_norm, dual_eclipsing_mass_sigma = normalized_poisson_error(dual_eclipsing_mass, M_bins)


primary_mass = primary_population[:, 0]

mass_bin_centers, primary_mass_norm, primary_mass_sigma = normalized_poisson_error(general_mass, M_bins)



secondary_mass = secondary_population[:, 0]

mass_bin_centers, secondary_mass_norm, secondary_mass_sigma = normalized_poisson_error(secondary_mass, M_bins)



plt.hist(dual_eclipsing_mass, bins =  M_bins, density = True, color = 'green',
         alpha = 0.5, label = 'Dual Eclipsing Population')

plt.errorbar(mass_bin_centers, dual_eclipsing_mass_norm, yerr = dual_eclipsing_mass_sigma,
             color = 'green', linestyle = '', capsize = 4, alpha = 0.5)



plt.hist(primary_population[:, 0], bins = M_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Primary Population')

plt.errorbar(mass_bin_centers, primary_mass_norm, yerr = primary_mass_sigma,
             color = 'red', linestyle = '', capsize = 4, alpha = 0.5)



plt.hist(secondary_population[:, 0], bins =  M_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'Secondary Population')

plt.errorbar(mass_bin_centers, secondary_mass_norm, yerr = secondary_mass_sigma,
             color = 'blue', linestyle = '', capsize = 4, alpha = 0.5)

plt.xlabel(r'Primary Mass ($M_{\odot}$)', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()



##Plotting ass

dual_eclipsing_q = dual_eclipsing_population[:, 1]/dual_eclipsing_population[:, 0]

q_bin_centers, dual_eclipsing_q_norm, dual_eclipsing_q_sigma = normalized_poisson_error(dual_eclipsing_q, q_bins)




primary_q = primary_population[:, 1]/primary_population[:, 0]

q_bin_centers, primary_q_norm, primary_q_sigma = normalized_poisson_error(primary_q, q_bins)





secondary_q = secondary_population[:, 1]/secondary_population[:, 0]

q_bin_centers, secondary_q_norm, secondary_q_sigma = normalized_poisson_error(secondary_q, q_bins)




plt.hist(dual_eclipsing_q, bins = q_bins, density = True, color = 'green',
         alpha = 0.5, label = 'Dual Eclipsing Population')


plt.errorbar(q_bin_centers, dual_eclipsing_q_norm, yerr = dual_eclipsing_q_sigma,
             color = 'green', linestyle = '', capsize = 4, alpha = 0.5)



plt.hist(primary_q, bins = q_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Primary Population')


plt.errorbar(q_bin_centers, primary_q_norm, yerr = primary_q_sigma,
             color = 'red', linestyle = '', capsize = 4, alpha = 0.5)



plt.hist(secondary_q, bins =  q_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'Secondary Population')

plt.errorbar(q_bin_centers, secondary_q_norm, yerr = secondary_q_sigma,
             color = 'blue', linestyle = '', capsize = 4, alpha = 0.5)

plt.xlabel('Mass Ratio q', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()



##Plotting log P

dual_eclipsing_log_P = dual_eclipsing_population[:, 4]

log_P_bin_centers, dual_eclipsing_log_P_norm, dual_eclipsing_log_P_sigma = normalized_poisson_error(dual_eclipsing_log_P, log_P_bins)




primary_log_P = primary_population[:, 4]

log_P_bin_centers, primary_log_P_norm, primary_log_P_sigma = normalized_poisson_error(primary_log_P, log_P_bins)





secondary_log_P = secondary_population[:, 4]

log_P_bin_centers, secondary_log_P_norm, secondary_log_P_sigma = normalized_poisson_error(secondary_log_P, log_P_bins)






         
plt.hist(dual_eclipsing_log_P, bins =  log_P_bins, density = True, color = 'green',
         alpha = 0.5, label = 'Dual Eclipsing Population')

plt.errorbar(log_P_bin_centers, dual_eclipsing_log_P_norm, yerr = dual_eclipsing_log_P_sigma,
             color = 'green', linestyle = '', capsize = 4, alpha = 0.5)



plt.hist(primary_log_P, bins = log_P_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Primary Population')

plt.errorbar(log_P_bin_centers, primary_log_P_norm, yerr = primary_log_P_sigma,
             color = 'red', linestyle = '', capsize = 4, alpha = 0.5)




plt.hist(secondary_log_P, bins =  log_P_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'Secondary Population')

plt.errorbar(log_P_bin_centers, secondary_log_P_norm, yerr = secondary_log_P_sigma,
             color = 'blue', linestyle = '', capsize = 4, alpha = 0.5)

plt.xlabel(r'log(P [Days])', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()



#Plotting Eccentricity


dual_eclipsing_ecc = dual_eclipsing_population[:, 5]

ecc_bin_centers, dual_eclipsing_ecc_norm, dual_eclipsing_ecc_sigma = normalized_poisson_error(dual_eclipsing_ecc, ecc_bins)




primary_ecc = primary_population[:, 5]

ecc_bin_centers, primary_ecc_norm, primary_ecc_sigma = normalized_poisson_error(primary_ecc, ecc_bins)





secondary_ecc = secondary_population[:, 5]

ecc_bin_centers, secondary_ecc_norm, secondary_ecc_sigma = normalized_poisson_error(secondary_ecc, ecc_bins)

         
plt.hist(dual_eclipsing_ecc, bins =  ecc_bins, density = True, color = 'green',
         alpha = 0.5, label = 'Dual Eclipsing Population')

plt.errorbar(ecc_bin_centers, dual_eclipsing_ecc_norm, yerr = dual_eclipsing_ecc_sigma,
             color = 'green', linestyle = '', capsize = 4, alpha = 0.5)


plt.hist(primary_ecc, bins = ecc_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Primary Population')

plt.errorbar(ecc_bin_centers, primary_ecc_norm, yerr = primary_ecc_sigma,
             color = 'red', linestyle = '', capsize = 4, alpha = 0.5)


plt.hist(secondary_ecc, bins =  ecc_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'Secondary Population')

plt.errorbar(ecc_bin_centers, secondary_ecc_norm, yerr = secondary_ecc_sigma,
             color = 'blue', linestyle = '', capsize = 4, alpha = 0.5)



plt.xlabel(r'Eccentricity', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()



#Plotting ω


dual_eclipsing_omega = dual_eclipsing_population[:, 6]

omega_bin_centers, dual_eclipsing_omega_norm, dual_eclipsing_omega_sigma = normalized_poisson_error(dual_eclipsing_omega, omega_bins)



primary_omega = primary_population[:, 6]

omega_bin_centers, primary_omega_norm, primary_omega_sigma = normalized_poisson_error(primary_omega, omega_bins)



secondary_omega = secondary_population[:, 6]

omega_bin_centers, secondary_omega_norm, secondary_omega_sigma = normalized_poisson_error(secondary_omega, omega_bins)



         
plt.hist(dual_eclipsing_omega, bins =  omega_bins, density = True, color = 'green',
         alpha = 0.5, label = 'Dual Eclipsing Population')

plt.errorbar(omega_bin_centers, dual_eclipsing_omega_norm, yerr = dual_eclipsing_omega_sigma,
             color = 'green', linestyle = '', capsize = 4, alpha = 0.5)



plt.hist(primary_omega, bins = omega_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Primary Population')

plt.errorbar(omega_bin_centers, primary_omega_norm, yerr = primary_omega_sigma,
             color = 'red', linestyle = '', capsize = 4, alpha = 0.5)



plt.hist(secondary_omega, bins =  omega_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'Secondary Population')

plt.errorbar(omega_bin_centers, secondary_omega_norm, yerr = secondary_omega_sigma,
             color = 'blue', linestyle = '', capsize = 4, alpha = 0.5)


plt.xlabel(r'ω', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()



##Plotting Ω

dual_eclipsing_Omega = dual_eclipsing_population[:, 7]

Omega_bin_centers, dual_eclipsing_Omega_norm, dual_eclipsing_Omega_sigma = normalized_poisson_error(dual_eclipsing_Omega, Omega_bins)


primary_Omega = primary_population[:, 7]

Omega_bin_centers, primary_Omega_norm, primary_Omega_sigma = normalized_poisson_error(primary_Omega, Omega_bins)


secondary_Omega = secondary_population[:, 7]

Omega_bin_centers, secondary_Omega_norm, secondary_Omega_sigma = normalized_poisson_error(secondary_Omega, Omega_bins)



         
plt.hist(dual_eclipsing_Omega, bins =  Omega_bins, density = True, color = 'green',
         alpha = 0.5, label = 'Dual Eclipsing Population')

plt.errorbar(Omega_bin_centers, dual_eclipsing_Omega_norm, yerr = dual_eclipsing_Omega_sigma,
             color = 'green', linestyle = '', capsize = 4, alpha = 0.5)



plt.hist(primary_Omega, bins = Omega_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Primary Population')

plt.errorbar(Omega_bin_centers, primary_Omega_norm, yerr = primary_Omega_sigma,
             color = 'red', linestyle = '', capsize = 4, alpha = 0.5)



plt.hist(secondary_Omega, bins =  Omega_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'Secondary Population')

plt.errorbar(Omega_bin_centers, secondary_Omega_norm, yerr = secondary_Omega_sigma,
             color = 'blue', linestyle = '', capsize = 4, alpha = 0.5)

plt.xlabel(r'Ω', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()



##Plotting cos(i)

dual_eclipsing_cos_i = dual_eclipsing_population[:, 8]

cos_i_bin_centers, dual_eclipsing_cos_i_norm, dual_eclipsing_cos_i_sigma = normalized_poisson_error(dual_eclipsing_cos_i, cos_i_bins)


primary_cos_i = primary_population[:, 8]

cos_i_bin_centers, primary_cos_i_norm, primary_cos_i_sigma = normalized_poisson_error(primary_cos_i, cos_i_bins)


secondary_cos_i = secondary_population[:, 8]

cos_i_bin_centers, secondary_cos_i_norm, secondary_cos_i_sigma = normalized_poisson_error(secondary_cos_i, cos_i_bins)


         
plt.hist(dual_eclipsing_cos_i, bins =  cos_i_bins, density = True, color = 'green',
         alpha = 0.5, label = 'Dual Eclipsing Population')

plt.errorbar(cos_i_bin_centers, dual_eclipsing_cos_i_norm, yerr = dual_eclipsing_cos_i_sigma,
             color = 'green', linestyle = '', capsize = 4, alpha = 0.5)



plt.hist(primary_cos_i, bins = cos_i_bins, density = True,
         color = 'red', alpha = 0.5, label = 'Primary Population')

plt.errorbar(cos_i_bin_centers, primary_cos_i_norm, yerr = primary_cos_i_sigma,
             color = 'red', linestyle = '', capsize = 4, alpha = 0.5)



plt.hist(secondary_cos_i, bins =  cos_i_bins, density = True, color = 'blue',
         alpha = 0.5, label = 'Secondary Population')

plt.errorbar(cos_i_bin_centers, secondary_cos_i_norm, yerr = secondary_cos_i_sigma,
             color = 'blue', linestyle = '', capsize = 4, alpha = 0.5)



plt.xlabel(r'cos(i)', fontsize = 18)
plt.ylabel('PDF', fontsize = 18)
plt.tick_params(direction = 'in', labelsize = 14)
plt.legend(fontsize = 16)
plt.show()

From these graphs, we can see interesting differences between our 'dual\-eclipsing', 'primary only' and 'secondary only' populations. Primary Mass, Mass Ratio q do not have significant differences, while Ω and the angle of inclination I peak at the same points as that on the eclipsing population vs binary systems population histograms.

## Eccentricity

The eccentricity distributions for these three subcategories of eclipses are very different! For the dual\-eclipsing population, their orbits are mainly circular; though for the primary/secondary only population, their orbits are very eccentric. Systems with low eccentricity have a very high chance to have a short period, and if they have a short period then the stars are very close to one another, thus a primary AND a secondary eclipse is unavoidable. For there to be only one eclipse, the orbit has to be eccentric, since the pathway of the secondary star in front of the primary star is vasstly different from the pathway behind the primary star from our reference point.

## Period

Similarly, as we said that the dual\-eclipsing stars mainly have circular orbits, that also means that they would have lower periods. This is shown on the graph, which essentialyl verifies our explanation for the eccentricity distributions.

## ω

This parameter here is the most mind\-boggling! How can a parameter that barely affected eclipses in the general eclipsing population have such a big influence here? This is because a primary or secondary eclipse is determined by where the periapse is from our reference point. In one respect the periapse represents the most desirable point in 3d space where we'd likely see an eclipse because the two stars are physically close to one another. If ω is close to 0 or 2π then that desirable point will lie between the observer and the primary star, great conditions for a primary eclipse. If ω is close to π then the primary star will be between the observer and the periapse, so an eclipse near the periapse will be seen as a secondary eclipse.



# Conclusions

Our little tutorial about how to generate binary systems and find eclipses has shown very interesting results. By comparing the systems that have eclipses versus the binary population, we can find patterns that show how eclipsing systems differ geometrically. We found that parameters like the mass of the primary star, the mass of the secondary star and ω do not affect the eclipsing rate since their distributions look the same no matter what. However, eccentricity matters–most eclipses are either almost perfectly circular and have short periods, or are very eccentric. This consequentially affects the distribution of periods within our eclipsing population, as systems with eclipses generally have shorter periods. Ω and the angle of inclination I both peak at 0,π and 2π.

We also compared primary and secondary eclipses. From here we found that systems with both primary and secondary eclipses mainly have circular orbits, while systems with only either a primary or a secondary eclipse have eccentric ones. And subsequently, these dual\-eclipsing systems also have shorter orbital periods. We also discovered that the angular parameter ω affects whether an eclipsing system has both eclipses or just one since it depends on where the periapse is located!

Hope you enjoyed our tutorial! Please also take the time to enjoy this hilarious joke about one of the closest binary star systems to Earth.
